Step 1: Dataset Selection and Loading

In [1]:
import pandas as pd
data = pd.read_csv("data/telco_churn.csv")
print(data.shape)
data.head()

(7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


Step 2a: Data Understanding

In [2]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

In [3]:
print(data.isnull().sum())
print("Duplicate rows:", data.duplicated().sum())

customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
Duplicate rows: 0


In [4]:
print(data["Churn"].value_counts())
print((data["Churn"].value_counts(normalize=True) * 100).round(2))

Churn
No     5174
Yes    1869
Name: count, dtype: int64
Churn
No     73.46
Yes    26.54
Name: proportion, dtype: float64


In [5]:
data["TotalCharges"] = pd.to_numeric(data["TotalCharges"], errors="coerce")
print(data["TotalCharges"].isnull().sum())
data[data["TotalCharges"].isnull()][["customerID", "tenure", "TotalCharges"]]

11


,customerID,tenure,TotalCharges
488,4472-LVYGI,0,NaN
753,3115-CZMZD,0,NaN
936,5709-LVOEQ,0,NaN
1082,4367-NUYAO,0,NaN
1340,1371-DWPAZ,0,NaN
3331,7644-OMVMY,0,NaN
3826,3213-VVOLG,0,NaN
4380,2520-SGTTA,0,NaN
5218,2923-ARZLG,0,NaN
6670,4075-WKNIU,0,NaN


preprocessing and split

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X = data.drop(columns=["customerID", "Churn"])
y = data["Churn"].map({"No": 0, "Yes": 1})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]
cat_cols = [c for c in X.columns if c not in num_cols]   # SeniorCitizen goes here too

In [7]:
numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value=0)),
    ("scale", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols)
])

In [8]:
print(X_train.shape, X_test.shape)
print(y_train.value_counts(normalize=True).round(4))
print(y_test.value_counts(normalize=True).round(4))

(5634, 19) (1409, 19)
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64
Churn
0    0.7346
1    0.2654
Name: proportion, dtype: float64


In [9]:
# ---- Imports: the five classifiers and the evaluation metrics ----
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, roc_auc_score, precision_score,
                             recall_score, f1_score, matthews_corrcoef)

# ---- The five models to compare (fixed random_state so results are reproducible) ----
classifiers = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
}

trained_models = {}   # stores each fitted pipeline (needed later for saving/verification)
rows = []             # stores one row of metrics per model

for name, clf in classifiers.items():
    # Pipeline = preprocessing + classifier. clone() gives every model its own copy
    # of the preprocessor so the models do not share fitted state.
    pipe = Pipeline([("prep", clone(preprocessor)), ("clf", clf)])

    # Fit on the TRAINING data only: scaler/encoder statistics come from X_train alone
    pipe.fit(X_train, y_train)
    trained_models[name] = pipe

    # Hard predictions (0 = stayed, 1 = churned) for accuracy/precision/recall/F1/MCC
    pred = pipe.predict(X_test)

    # Predicted probability of churn (class 1), needed for the AUC score
    prob = pipe.predict_proba(X_test)[:, 1]

    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),           # overall correct predictions
        "AUC": roc_auc_score(y_test, prob),                 # ranking quality across thresholds
        "Precision": precision_score(y_test, pred, zero_division=0),  # of predicted churners, how many were real
        "Recall": recall_score(y_test, pred, zero_division=0),        # of real churners, how many were found
        "F1": f1_score(y_test, pred, zero_division=0),      # balance of precision and recall
        "MCC": matthews_corrcoef(y_test, pred),             # balanced score, robust to class imbalance
    })

# Combine into one comparison table, rounded to 4 decimals
results = pd.DataFrame(rows).round(4)
results

,Model,Accuracy,AUC,Precision,Recall,F1,MCC
0,Logistic Regression,0.8055,0.8420,0.6572,0.5588,0.6040,0.4790
1,Decision Tree,0.7182,0.6415,0.4698,0.4786,0.4742,0.2818
2,KNN,0.7573,0.7882,0.5419,0.5535,0.5476,0.3818
3,Naive Bayes,0.6955,0.8075,0.4597,0.8396,0.5941,0.4268
4,Random Forest,0.7828,0.8196,0.6164,0.4813,0.5405,0.4064
